# Train the small tactical model: Gemma 3 270M on your fights (free, in Google Colab)

The app's **small tactical model** answers, in a fight, with one of `ALL_IN, POKE, FARM, RETREAT, GROUP, RULES`.
Untrained, a 270M model guesses. This notebook teaches it **from your fights**: every fight choice the app made (the
Brain's, the small model's, or yours) is saved with the exact summary the model is shown and **how it went 4 s later**
(damage dealt, kills, HP kept, death). The model learns the choices that worked, and to leave when pushing on went badly.

Run every cell (*Runtime → Run all*). At the end you get **`gemma3-270m-mlbb-tactical.litertlm`** (also copied to
My Drive/MLBB-Research): put it in **Download/MLBB-Research** on the phone. The app puts it in the small-model slot by
itself (Model page → switch *Small tactical model in fights* on).

**Before you start**
1. App → Learn page → **Export fight moments (CSV)** (file `mlbb-fights-*.csv` in Downloads/MLBB-Research). A few
   matches give a few hundred moments; more is better (1000+).
2. Gemma needs a free Hugging Face account: open https://huggingface.co/google/gemma-3-270m-it , accept the licence, then
   make a token (huggingface.co → Settings → Access Tokens, type *Read*). In Colab, click the 🔑 (Secrets) on the left,
   add `HF_TOKEN` with the token, and allow this notebook to use it.
3. *Runtime → Change runtime type → T4 GPU* (training takes minutes on a GPU; the conversion runs on the CPU part).

Re-train every few matches: each run **continues from your last trained model** (kept in
My Drive/MLBB-Research/tactical_llm) and learns on all your fight moments again, so nothing is lost.

In [ ]:
# The converter to the phone's format (LiteRT-LM). PyTorch / transformers are already in Colab.
# (versions that were tested together: litert-torch 0.10.0 needs torch below 2.15 - pip picks it - and
#  litert-lm-builder 0.18.1; the newer 0.19 builder does not work with it. It also needs TensorFlow.)
!pip -q install "litert-torch==0.10.0" "litert-lm-builder==0.18.1"
import importlib.util, subprocess, sys
if importlib.util.find_spec("tensorflow") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow-cpu"], check=True)
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## 1. Hugging Face access (Gemma licence)

In [ ]:
from huggingface_hub import login
token = None
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception as e:
    print("No HF_TOKEN secret:", e)
if token:
    login(token=token); print("Logged in to Hugging Face with the HF_TOKEN secret")
else:
    print("Paste your Hugging Face token below (or add the HF_TOKEN secret and run this cell again)")
    login()

## 2. Load your fight moments
Easiest: put your `mlbb-fights-*.csv` files in **My Drive/MLBB-Research** - the cell takes them from there. Otherwise it
asks you to pick them on this device. Several files are fine (the same moment in two exports counts once).

In [ ]:
import glob, os, re, random, numpy as np, pandas as pd
from google.colab import files
DRIVE_DIR = "/content/drive/MyDrive/MLBB-Research"
paths = []
try:
    from google.colab import drive
    drive.mount("/content/drive")
    paths = sorted(glob.glob(f"{DRIVE_DIR}/mlbb-fights-*.csv"))
except Exception as e:
    print("Google Drive not used:", e)
if paths:
    print(f"Using {len(paths)} files from My Drive/MLBB-Research:", ", ".join(os.path.basename(p) for p in paths))
else:
    print("No mlbb-fights-*.csv in My Drive/MLBB-Research - pick them on this device:")
    up = files.upload()
    paths = [os.path.abspath(k) for k in up if k.endswith(".csv")]
df = pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)
df = df.drop_duplicates(subset=["ts", "option", "summary"]).sort_values("ts").reset_index(drop=True)
df["summary"] = df["summary"].astype(str)
print(len(df), "fight moments;", "chosen by you:", int(df.by_you.sum()), "| deaths:", int(df.died.sum()))
print(df.groupby("option").reward.agg(["count", "mean"]).round(2))

## 3. What the model should have answered
- a choice that **went well** (score +0.3 or more) → that choice is the answer (your own good choices count twice)
- **your** choice that was not a disaster → that choice (it learns your style)
- a choice that **went badly** (score −0.5 or less, or you died) and pushed on (all in, poke, farm, rules) → the answer
  is to leave: **GROUP** when allies were close, else **RETREAT**
- everything else (unclear outcome) is left out
No option may be more than 35% of the examples (so it does not learn "always say the same thing"). The newest 15% of
moments are kept aside to test it.

In [ ]:
OPTIONS = ["ALL_IN", "POKE", "FARM", "RETREAT", "GROUP", "RULES"]
GOOD, BAD = 0.3, -0.5

# EXACTLY the question the app asks (ai/TacticalAdvisor.kt: prompt)
def prompt(summary):
    return ("You play Mobile Legends: Bang Bang. Choose the best action for this fight moment.\n" + summary +
            "\nOptions: ALL_IN, POKE, FARM, RETREAT, GROUP, RULES\nAnswer with one option only.")

def allies_close(summary):
    m = re.search(r"allies close: (\d+)", summary)
    return int(m.group(1)) if m else 0

def label(row):
    o, r, mine, died = row.option, float(row.reward), int(row.by_you) == 1, int(row.died) == 1
    if o not in OPTIONS: return None, 0
    if r >= GOOD and not died: return o, (2 if mine else 1)
    if mine and r > BAD and not died: return o, 1
    if (r <= BAD or died) and o in ("ALL_IN", "POKE", "FARM", "RULES"):
        return ("GROUP" if allies_close(row.summary) > 0 else "RETREAT"), 1
    return None, 0

rows = []
for i, row in df.iterrows():
    a, n = label(row)
    if a: rows += [(i, row.summary, a)] * n
ex = pd.DataFrame(rows, columns=["i", "summary", "answer"])
split = df.index[int(len(df) * 0.85)] if len(df) > 20 else len(df)
train, test = ex[ex.i < split], ex[ex.i >= split].drop_duplicates("i")
# no answer above 35% of the training examples
cap = max(50, int(0.35 * len(train)))
train = pd.concat([g.sample(min(len(g), cap), random_state=0) for _, g in train.groupby("answer")]).sample(frac=1, random_state=0)
print("training examples:", len(train), "| test moments:", len(test))
print(train.answer.value_counts())
assert len(train) >= 150, ("Too few usable fight moments (" + str(len(train)) + "). Play a few more matches with the AI "
                           "(or yourself with learning on), export again, and re-run.")

## 4. The model: your last trained one if there is one, else Gemma 3 270M

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
BASE = "google/gemma-3-270m-it"
SAVED = f"{DRIVE_DIR}/tactical_llm"          # your last trained model (Drive)
start = SAVED if os.path.exists(os.path.join(SAVED, "config.json")) else BASE
print("Starting from:", "your last trained model (continues learning)" if start == SAVED else BASE)
tok = AutoTokenizer.from_pretrained(start)
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
dt = torch.bfloat16 if bf16 else torch.float32
try:    # (newer transformers call it dtype, older torch_dtype)
    model = AutoModelForCausalLM.from_pretrained(start, dtype=dt, attn_implementation="eager")
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(start, torch_dtype=dt, attn_implementation="eager")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
END = "<end_of_turn>"

## 5. Train
Only the answer is learned (the question is context). A few minutes on a T4.

In [ ]:
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

def encode(summary, answer):
    chat = tok.apply_chat_template([{"role": "user", "content": prompt(summary)}], add_generation_prompt=True, tokenize=False)
    p = tok(chat, add_special_tokens=False)["input_ids"]        # (the template already holds <bos>)
    a = tok(answer + END, add_special_tokens=False)["input_ids"]
    ids = (p + a)[-448:]
    labels = ([-100] * len(p) + a)[-448:]
    return {"input_ids": ids, "labels": labels}

class Fights(Dataset):
    def __init__(self, d): self.items = [encode(s, a) for s, a in zip(d.summary, d.answer)]
    def __len__(self): return len(self.items)
    def __getitem__(self, i): return self.items[i]

def collate(batch):
    n = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    ids = torch.full((len(batch), n), pad); lab = torch.full((len(batch), n), -100); att = torch.zeros((len(batch), n), dtype=torch.long)
    for k, b in enumerate(batch):
        L = len(b["input_ids"])
        ids[k, :L] = torch.tensor(b["input_ids"]); lab[k, :L] = torch.tensor(b["labels"]); att[k, :L] = 1
    return {"input_ids": ids, "labels": lab, "attention_mask": att}

epochs = 3 if len(train) < 3000 else 2
steps = -(-len(train) // 16) * epochs
args = TrainingArguments(output_dir="/content/ckpt", per_device_train_batch_size=16, gradient_accumulation_steps=1,
                         num_train_epochs=epochs, learning_rate=5e-5 if start == BASE else 2e-5, warmup_steps=max(1, steps // 20),
                         lr_scheduler_type="cosine", logging_steps=20, save_strategy="no", report_to=[],
                         bf16=bf16, fp16=False)
Trainer(model=model, args=args, train_dataset=Fights(train), data_collator=collate).train()

## 6. How good is it? (on your newest fights, which it did not train on)
It is compared with always giving the most common answer. It only has to be *useful advice*: the app still checks every
answer against its scores and safety rules.

In [ ]:
def parse(reply):                       # the same reading as the app (TacticalAdvisor.parse)
    u = reply.upper().replace("-", "_").replace(" ", "_")
    hits = [(u.find(o), o) for o in OPTIONS if u.find(o) >= 0]
    return min(hits)[1] if hits else ("ALL_IN" if "ALLIN" in u else None)

@torch.no_grad()
def ask(summaries, bs=32):
    model.eval(); tok.padding_side = "left"; out = []
    for k in range(0, len(summaries), bs):
        chats = [tok.apply_chat_template([{"role": "user", "content": prompt(s)}], add_generation_prompt=True, tokenize=False)
                 for s in summaries[k:k + bs]]
        enc = tok(chats, return_tensors="pt", padding=True, add_special_tokens=False).to(device)
        gen = model.generate(**enc, max_new_tokens=8, do_sample=False)
        out += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    tok.padding_side = "right"; return out

if len(test):
    replies = ask(list(test.summary))
    pred = [parse(r) for r in replies]
    acc = float(np.mean([p == a for p, a in zip(pred, test.answer)]))
    base = float((test.answer == train.answer.value_counts().idxmax()).mean())
    print(f"right answer: {acc*100:.0f}%   (always the most common answer: {base*100:.0f}%)")
    print("answers it gave:", pd.Series(pred).value_counts(dropna=False).to_dict())
    print("\nexample:\n" + prompt(test.summary.iloc[0]) + "\n->", replies[0], "| wanted:", test.answer.iloc[0])

## 7. Keep it (for the next run) and convert it for the phone

In [ ]:
import shutil
OUT_HF = "/content/tactical_llm"
model.to(torch.float32).save_pretrained(OUT_HF, safe_serialization=True)
tok.save_pretrained(OUT_HF)
if os.path.isdir("/content/drive/MyDrive"):
    shutil.rmtree(SAVED, ignore_errors=True); shutil.copytree(OUT_HF, SAVED)
    print("Saved in My Drive/MLBB-Research/tactical_llm (the next run continues from it)")

In [ ]:
# Convert to the phone's format: int8 weights, prompts up to 256 tokens, 512 tokens of memory (small and fast).
# The model's own chat template goes into the file, so the app's questions look exactly like the training ones.
import gc; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
OUT = "/content/litert_out"; shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT)
NAME = "gemma3-270m-mlbb-tactical.litertlm"
try:
    from litert_torch.generative.export_hf import export as hf_export
    hf_export.export(model=OUT_HF, output_dir=OUT, prefill_lengths=[256], cache_length=512,
                     quantization_recipe="dynamic_wi8_afp32")
except Exception as e:
    # older route: the Gemma 3 270M example converter, with the Gemma chat format written in by hand
    print("HF export did not work (", e, ") - using the Gemma 3 converter")
    from litert_torch.generative.examples.gemma3 import gemma3
    from litert_torch.generative.utilities import converter, export_config as ec
    from litert_torch.generative.layers import kv_cache as kv_utils
    m = gemma3.build_model_270m(OUT_HF, mask_cache_size=0)
    cfg = ec.ExportConfig(); cfg.mask_as_input = True; cfg.kvcache_layout = kv_utils.KV_LAYOUT_TRANSPOSED
    converter.convert_to_litert(m, output_path=OUT, output_name_prefix="gemma3-270m-mlbb-tactical",
        prefill_seq_len=[256], kv_cache_max_len=512, quantize="dynamic_int8", export_config=cfg, output_format="litertlm",
        hf_tokenizer_model_path=os.path.join(OUT_HF, "tokenizer.json"), start_token="<bos>",
        stop_tokens=["<end_of_turn>", "<eos>"], user_prompt_prefix="<start_of_turn>user\n", user_prompt_suffix="<end_of_turn>\n",
        model_prompt_prefix="<start_of_turn>model\n", model_prompt_suffix="<end_of_turn>\n", llm_model_type="gemma3")
made = sorted(glob.glob(f"{OUT}/**/*.litertlm", recursive=True), key=os.path.getmtime)
assert made, "No .litertlm file was made - see the messages above"
final = f"/content/{NAME}"; shutil.copy(made[-1], final)
print(NAME, f"{os.path.getsize(final)/1e6:.0f} MB")

## 8. Get the file

In [ ]:
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(DRIVE_DIR, exist_ok=True); shutil.copy(final, f"{DRIVE_DIR}/{NAME}")
    print("Copied to My Drive/MLBB-Research/" + NAME + " - download it from Drive to the phone's Download/MLBB-Research")
files.download(final)

**On the phone:** put `gemma3-270m-mlbb-tactical.litertlm` in **Download/MLBB-Research** → app → Model page →
*Scan the folder now* (or restart the app) → switch **Small tactical model in fights** on. The Model page shows how often
its answers are used or rejected by the app's checks; fewer rejections after training = it learned.